# eRisk Notebook

In [4]:
from __future__ import annotations

import json
import os
from pathlib import Path

from app.cli import run_eval

OUT = Path('outputs')
OUT.mkdir(exist_ok=True)


In [5]:
cfg = {
    'persona_count': 10,
    'seed': 42,
    'eval_mode': 'synthetic_only',
    'prompt_version': os.getenv('PROMPT_VERSION', 'v1'),
    'save_diagnostics': True,
    'max_api_calls': 180,
    'trace_level': 'compact',
    'fit_calibrator_policy': 'auto',
}
cfg


{'persona_count': 10,
 'seed': 42,
 'eval_mode': 'synthetic_only',
 'prompt_version': 'v1',
 'save_diagnostics': True,
 'max_api_calls': 180,
 'trace_level': 'compact',
 'fit_calibrator_policy': 'auto'}

In [6]:
# Run one eval iteration (same engine as CLI)
run_eval(**cfg)


--- Eval Mode: synthetic_only | personas=10 | seed=42 | prompts=v1 ---
Backend info: auto_switch=on | cuda_available=False | vram_gb=0.00 | min_vram_gb=8.00 | cuda_gate=fail
Resolved backends: detector=openrouter [meta-llama/llama-3-8b-instruct] | persona=openrouter_sim [meta-llama/llama-3-8b-instruct]
Runtime controls: trace_level=compact | max_api_calls=180
Calibrator policy: requested=auto | enabled=False | min_train_records=10
Synthetic generator: version=sim_v2 | strict_split_lock=on

=== Persona 1 (synthetic/val/risk_leaning) ===
Evaluation calls=21/180 [######------------------] 1/4
=== Persona 8 (synthetic/val/somatic_evasive) ===
Evaluation calls=41/180 [############------------] 2/4
=== Persona 2 (synthetic/test/somatic_evasive) ===
Evaluation calls=58/180 [##################------] 3/4
=== Persona 4 (synthetic/test/cognitive_ruminative) ===
Evaluation calls=79/180 [########################] 4/4

--- Evaluation Summary ---
Primary split: overall_labeled | binary_f1=0.4000 | b

In [7]:
def load_json(name: str):
    return json.loads((OUT / name).read_text(encoding='utf-8'))

metrics = load_json('metrics_run_local.json')
leakage = load_json('leakage_report_run_local.json')
manifest = load_json('persona_manifest_run_local.json')

{
    'primary_eval_split': metrics.get('primary_eval_split'),
    'primary_metrics': metrics.get('primary_metrics', {}),
    'strict_pass': leakage.get('strict_pass'),
    'leakage_failures': leakage.get('failure_reasons', []),
}


{'primary_eval_split': 'overall_labeled',
 'primary_metrics': {'binary_accuracy': 0.25,
  'binary_f1': 0.4,
  'bdi_mae': 10.5,
  'symptom_f1_at_4': 0.25,
  'avg_turns_to_decision': 10.0,
  'risk_recall': 0.0,
  'objective': 0.25},
 'strict_pass': True,
 'leakage_failures': []}

In [8]:
{
    'split_sizes': leakage.get('split_sizes', {}),
    'id_overlap_counts': leakage.get('id_overlap_counts', {}),
    'template_overlap_counts': leakage.get('template_overlap_counts', {}),
    'manifest_hash': leakage.get('manifest_hash'),
}


{'split_sizes': {'train': 6, 'val': 2, 'test': 2},
 'id_overlap_counts': {'train_val': 0, 'train_test': 0, 'val_test': 0},
 'template_overlap_counts': {'train_val': 0, 'train_test': 0, 'val_test': 0},
 'manifest_hash': '4adc376ff9717a184e355c61384ba912cad784411dfc1eb8f34235f5affaf319'}

In [9]:
profiles = manifest.get('profiles', [])
family_counts = {}
for p in profiles:
    fam = p.get('family', 'unknown')
    family_counts[fam] = family_counts.get(fam, 0) + 1

{
    'profile_count': len(profiles),
    'family_counts': family_counts,
}


{'profile_count': 10,
 'family_counts': {'risk_leaning': 2,
  'somatic_evasive': 3,
  'control_stressed': 1,
  'control_neutral': 1,
  'cognitive_ruminative': 2,
  'mixed_moderate': 1}}

In [10]:
failure = load_json('failure_report_run_local.json')
{
    'family_count': failure.get('family_count', {}),
    'binary_f1_by_family': failure.get('binary_f1_by_family', {}),
    'bdi_mae_by_family': failure.get('bdi_mae_by_family', {}),
}


{'family_count': {'risk_leaning': 1,
  'somatic_evasive': 2,
  'cognitive_ruminative': 1},
 'binary_f1_by_family': {'risk_leaning': 0.0,
  'somatic_evasive': 0.6667,
  'cognitive_ruminative': 0.0},
 'bdi_mae_by_family': {'risk_leaning': 16.0,
  'somatic_evasive': 5.0,
  'cognitive_ruminative': 16.0}}

In [11]:
diagnostics = load_json('diagnostics_run_local.json')

summary_rows = []
for row in diagnostics:
    fs = row.get('final_state', {})
    mi = fs.get('module_imputation', {})
    summary_rows.append({
        'persona': row.get('LLM'),
        'raw_label': fs.get('raw_predicted_label'),
        'raw_bdi': fs.get('raw_predicted_bdi_score'),
        'final_label': fs.get('predicted_label'),
        'final_bdi': fs.get('predicted_bdi_score'),
        'imputed_items': mi.get('imputed_item_count', 0),
    })

summary_rows


[{'persona': '1',
  'raw_label': 'control',
  'raw_bdi': 4,
  'final_label': 'control',
  'final_bdi': 5,
  'imputed_items': 3},
 {'persona': '8',
  'raw_label': 'control',
  'raw_bdi': 6,
  'final_label': 'control',
  'final_bdi': 7,
  'imputed_items': 4},
 {'persona': '2',
  'raw_label': 'depressed',
  'raw_bdi': 9,
  'final_label': 'depressed',
  'final_bdi': 5,
  'imputed_items': 0},
 {'persona': '4',
  'raw_label': 'control',
  'raw_bdi': 5,
  'final_label': 'control',
  'final_bdi': 5,
  'imputed_items': 3}]